In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [9]:
data = pd.read_csv("diabetes_prediction_dataset.csv")

# Encode categorical columns
le_gender = LabelEncoder()
le_smoke = LabelEncoder()

data["gender"] = le_gender.fit_transform(data["gender"])
data["smoking_history"] = le_smoke.fit_transform(data["smoking_history"])

# Split features and label
X = data.drop("diabetes", axis=1)
y = data["diabetes"]

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32).view(-1,1)

In [10]:
num_clients = 3

client_X = torch.chunk(X, num_clients)
client_y = torch.chunk(y, num_clients)

In [11]:
class DiabetesModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(8,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.network(x)

In [12]:
class DiabetesModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(8,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.network(x)

In [13]:
def federated_avg(weights):

    avg_weights = weights[0]

    for key in avg_weights.keys():

        for i in range(1,len(weights)):
            avg_weights[key] += weights[i][key]

        avg_weights[key] = avg_weights[key] / len(weights)

    return avg_weights

In [14]:
def local_train(model, X, y, epochs=5, lr=0.01):
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

    return model.state_dict()

In [15]:
global_model = DiabetesModel()

rounds = 20

for r in range(rounds):

    local_weights = []

    for i in range(num_clients):

        # send global model to client
        local_model = DiabetesModel()
        local_model.load_state_dict(global_model.state_dict())

        # local training
        weights = local_train(local_model, client_X[i], client_y[i])

        local_weights.append(weights)

    # server aggregates models
    global_weights = federated_avg(local_weights)

    global_model.load_state_dict(global_weights)

    print("Federated Round", r+1, "Completed")

Federated Round 1 Completed
Federated Round 2 Completed
Federated Round 3 Completed
Federated Round 4 Completed
Federated Round 5 Completed
Federated Round 6 Completed
Federated Round 7 Completed
Federated Round 8 Completed
Federated Round 9 Completed
Federated Round 10 Completed
Federated Round 11 Completed
Federated Round 12 Completed
Federated Round 13 Completed
Federated Round 14 Completed
Federated Round 15 Completed
Federated Round 16 Completed
Federated Round 17 Completed
Federated Round 18 Completed
Federated Round 19 Completed
Federated Round 20 Completed


In [16]:
with torch.no_grad():

    preds = global_model(X)

    preds = (preds > 0.5).float()

    accuracy = (preds == y).float().mean()

    print("\nFinal Model Accuracy:", accuracy.item())


Final Model Accuracy: 0.9150000214576721
